# DeepImpact (2021)
[[paper]](https://arxiv.org/abs/2104.12016)<br>

__DeepImpact__ — метод Learned Sparse Retrieval, который вместо статистических весов TF использует их вычисляемый аналог (называемый авторами impact score) а также использует Document Expansion - расширение описания документа релевантными вопросами.

__Постановка задачи__<br>
Решается классическая задача инфомрационного поиска. По заданному запросу $Q$ найти $K$ наиболее релевантных документов из коллекции $D = \{d_1, d_2, \ldots, d_N\}$

__Мотивация__<br>
Классический поиск (BM25) страдает от проблемы Vocabulary Mismatch: если в запросе "автомобиль", а в документе "машина", документ не будет найден. Методы Document Expansion (например, Doc2query) решают это добавлением новых слов, но используют стандартные формулы ранжирования (BM25), которые опираются на статистику частот (TF), а не на реальную семантическую значимость слова в конкретном контексте. 

__Существующие подходы__<br>
На момент 2021 года основными альтернативами были:
- BM25 (1990-е): использует частотные статистики, не учитывает семантику и синонимы.
- Doc2query / DocT5query (2019): расширяет документ предсказанными вопросами, но итоговое ранжирование все равно зависит от TF-IDF эвристик.
- Dense Retrieval (DPR, 2020): переводит тексты в плотные векторы. Требует специализированных векторных индексов (HNSW, FAISS), потребляет много оперативной памяти и часто проигрывает в точности на out-of-domain данных.
- DeepCT (2019): перевзвешивает существующие слова документа с помощью BERT, но не добавляет новые токены (нет расширения словаря).

__Идея__<br>
давайте модернизируем подход DeepCT - помимо нейросетевого вычисления весов TF давайте расширим описание документа релевантными токенами (сделаем Document Expansion) и кроме того обучать TF будем не под частоту токена, а прямо под релевантность

<img src="img/deepimpact/deepimpact1.png" width=500>

__Архитектура__<br>
DeepImpact состоит из следующих компонентов:
1. DocT5query: модуль обогащения документа дополнительными токенами, которые могут отсутствовать в исходном тексте
2. BERT-encoder: основа модели (обычно `BERT-base`)
3. Prediction Head: полносвязный MLP слой поверх выходных эмбеддингов BERT, который отображает вектор токена в одно скалярное значение (Impact Score)

__Алгоритм обучения__<br>
Модель обучается предсказывать семантическую значимость токенов на основе размеченных пар (запрос, документ):
1. Для пары $(Q, D)$ из обучающей выборки (например, MS MARCO) определяется вклад каждого токена $t \in D$, который также присутствует в запросе $Q$.
2. Модель минимизирует Cross-Entropy Loss между предсказанным скором и целевой меткой релевантности.
3. Особенность: модель учится присваивать высокие баллы тем словам, которые реально помогают "вытащить" документ по соответствующим запросам. Если слово часто встречается, но не несет смысла для поиска, его Impact Score будет низким.

__Индексация__
1. Документ расширяется с помощью DocT5query.
2. Все токены расширенного документа (оригинальные + сгенерированные) проходят через BERT.
3. Prediction Head вычисляет Impact Score $w$ для каждого уникального токена.
4. Создается инвертированный индекс, где вместо TF (частоты) хранится предсказанный $w$, квантованный в целое число.

__Алгоритм инференса__
1. Запрос $Q$ токенизируется.
2. Поиск в инвертированном индексе находит документы, содержащие токены запроса.
3. Score документа вычисляется как простая сумма импактов: $Score(Q, D) = \sum_{t \in Q \cap D} w_t$.
4. Отсутствие сложных логарифмов и нормализаций длины (как в BM25) ускоряет расчет итогового ранга.

__Результаты__<br>
Эксперименты проводились на коллекции MS MARCO Passage Ranking:
- по MRR@10
    - дает 0.326 против 0.187 у BM25
    - на 5.1 п.п. выше, чем у DocT5query
- быстрее методов Dense Retrieval засчет прямого запроса в инвертированный индекс
- лучше DeepCT

## Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [2]:
import torch
from transformers import BertTokenizer, BertModel, T5ForConditionalGeneration

# Шаг 1: Расширение документа с помощью DocT5query
def expand_document(doc_text):
    # Используем предобученную модель T5 для генерации дополнительных токенов
    model_name = "castorini/doc2query-t5-base-msmarco"
    tokenizer = BertTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name)

    input_ids = tokenizer.encode(doc_text, return_tensors="pt")
    outputs = model.generate(input_ids, max_length=64, num_return_sequences=1)
    expanded_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return expanded_text

# Шаг 2: Вычисление Impact Score для каждого токена
def compute_impact_scores(expanded_text):
    # Используем BERT для получения эмбеддингов токенов
    model_name = "bert-base-uncased"
    tokenizer = BertTokenizer.from_pretrained(model_name)
    model = BertModel.from_pretrained(model_name)

    input_ids = tokenizer.encode(expanded_text, return_tensors="pt")
    outputs = model(input_ids)
    token_embeddings = outputs.last_hidden_state

    # Пример: простая линейная проекция для получения Impact Score
    # В реальной реализации здесь будет обученный полносвязный слой
    impact_scores = torch.mean(token_embeddings, dim=2).squeeze()

    # Преобразуем в целые числа для инвертированного индекса
    impact_scores = (impact_scores * 100).int()
    
    return impact_scores

# Шаг 3: Создание инвертированного индекса
def create_inverted_index(expanded_text, impact_scores):
    tokens = expanded_text.split()
    inverted_index = {}
    
    for token, score in zip(tokens, impact_scores):
        if token not in inverted_index:
            inverted_index[token] = []
        inverted_index[token].append(score.item())
    
    return inverted_index

# Пример использования
document = "The car is fast and efficient."
expanded_document = expand_document(document)
impact_scores = compute_impact_scores(expanded_document)
inverted_index = create_inverted_index(expanded_document, impact_scores)

print("Expanded Document:", expanded_document)
print("Inverted Index:", inverted_index)

# Шаг 4: Поиск по запросу
def search(query, inverted_index):
    query_tokens = query.split()
    document_score = 0
    
    for token in query_tokens:
        if token in inverted_index:
            document_score += sum(inverted_index[token])
    
    return document_score

# Пример поиска
query = "fast car"
score = search(query, inverted_index)
print("Document Score for Query:", score)



ModuleNotFoundError: No module named 'torch'